# 04 · Calibrate: fit three optical parameters and check the uncertainties

A **calibration** turns data into parameter values: here, the argon's Rayleigh scattering and absorption lengths
and the reflectivity of the cathode's reflector foil, from the light of a 128 nm calibration flash (notebook 03). We do it as a *closure test*: make data from
known "true" values, fit, and check that the fit comes back to the truth within its quoted uncertainty. Run a closure
test before trusting a fit on real data.

The recipe, in four calls: a **model** (parameters to expected photoelectrons per sensor), **data**, a **space** (which
parameters to fit), a **fit**. Then: the uncertainty from the Fisher information, a profile scan, and a coverage
check over many pretend datasets.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import lumen
from lumen import tools
from lumen.compare import references
from lumen.tools import plot

WORK = Path(os.environ.get("LUMEN_NOTEBOOKS", "lumen-notebooks"))
WORK.mkdir(parents=True, exist_ok=True)
path = WORK / "sbnd.lumen"
if not path.exists():
    lumen.convert("lumen_tpc:sbnd", out=path, data=references.store())
det = lumen.load(path)
p0 = det.params()

flash = lumen.sources.Point((-1.0, 0.0, 2.5), n_photons=1e8, lam=128.0)


def model(p):
    """Expected photoelectrons per sensor: random numbers fixed, random choices made at the nominal values (reference),
    so a smooth function of p (notebook 03)."""
    return lumen.readout.pe(lumen.simulate(det, flash, p, samples=2**17, seed=0, reference=p0), p)

## Space, truth and data

Each parameter is one factor on its whole table (1 = the GDML's and LArSoft's values). The truth moves all three; the data are the
model at the truth with Poisson noise, as a real sensor count would be.

In [ ]:
RAY, ABS, REFL = "materials/LAr/RAYLEIGH", "materials/LAr/ABSLENGTH", "surfaces/Surface_vm2000/REFLECTIVITY"
space = tools.Space(p0, factors=[RAY, ABS, REFL])
truth = np.log([0.9, 1.1, 0.9])  # the space's vector holds each factor's logarithm
start = space.vector()  # every factor 1
data = np.random.default_rng(7).poisson(np.asarray(model(space.params(truth))))
print(f"{data.sum():,} photoelectrons in {data.size:,} sensors")

## Fit: first order and second order

The same problem, two solvers (notebook 03 showed both on a two-parameter landscape):

- `tools.descend`: first order, gradient steps (Adam);
- `tools.fit`: second order, Fisher scoring (Gauss-Newton steps with the Fisher information as the curvature).

In [ ]:
first = tools.descend(model, space, data, start, steps=40, lr=0.02)
second = tools.fit(model, space, data, start, steps=10)
plot.convergence(space, {"first order (Adam)": first, "second order (Fisher scoring)": second}, truth)
plt.show()

## Uncertainty from the Fisher covariance

A fit's covariance is the inverse Fisher information at the fitted values (`res.cov`; `res.sigma` gives each
parameter's standard error in its own units, here the factor). At the truth the same inverse is the Cramér-Rao
bound (notebook 03): no unbiased fit can do better.

In [ ]:
crb = np.linalg.inv(tools.fisher(model, space, truth))
bound = space.sigma(crb, truth)
true_v, start_v = space.values(truth), space.values(start)
print(f"{'parameter':40s} {'truth':>7s} {'start':>7s} {'fit':>7s} {'error':>7s} {'±CRB':>7s}")
for k in space.keys:
    fit_v = space.values(second.x)[k]
    print(f"{k:40s} {true_v[k]:7.4f} {start_v[k]:7.4f} {fit_v:7.4f} {second.sigma[k]:7.4f} {bound[k]:7.4f}")
plot.errors(space, {"first order": first, "second order": second}, truth, crb)
plt.show()

## A profile scan

The Fisher error assumes the likelihood is a parabola near its minimum. The **profile likelihood** checks it
without that assumption: fix one parameter at a series of values, refit the others each time, and see where the
loss rises by 1/2 above its minimum (the 1 sigma interval).

In [ ]:
v = space.values(second.x)[RAY]
values = v + second.sigma[RAY] * np.linspace(-2.5, 2.5, 7)
pr = tools.profile(model, space, data, RAY, values, x0=second.x, steps=3)
plot.profile(pr, fit=second, space=space)
plt.show()

## Coverage over toys

One closure test can be lucky. Repeat it: draw new pretend datasets from the truth (*toys*), fit each, and count how
often the truth lies within the fit's ±1 sigma. For honest errors that is 68 % of the time, and the pulls
(fit - truth) / sigma follow a normal distribution of width 1. With 20 toys the count itself is uncertain by about
±10 %.

In [ ]:
cov = tools.coverage(model, space, truth, toys=20, seed=11, steps=3)
plot.pulls(cov, space)
plt.show()
print("pull mean", np.round(cov.pulls.mean(0), 2), " pull width", np.round(cov.pulls.std(0, ddof=1), 2))

## Keep the result

A calibration is saved by name beside the detector file, with only the values that changed; the detector file
itself is untouched. Loading with `calibration=` applies it.

In [ ]:
saved = det.save_calibration(second.params, "argon_fit", directory=path / "calibrations",
                             note="closure test of notebook 04")
again = lumen.load(path, calibration="argon_fit")
print(saved.name, "->", again.params().explain(RAY))